# 07 — Evaluation and Benchmark Comparison

This notebook loads the saved accuracy matrices from all five methods and generates the
complete set of comparison figures:

1. **Accuracy matrix heatmap** per method
2. **Forgetting curves** — accuracy on Task 0 as tasks accumulate
3. **BWT / FWT bar chart** — backward and forward transfer comparison
4. **Sparsity vs accuracy scatter** — efficiency frontier
5. **4-panel benchmark comparison** — the hero figure

Run all five training notebooks (02–06) before running this one.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from pathlib import Path

from src.evaluate import backward_transfer, forward_transfer, average_accuracy

%matplotlib inline
plt.rcParams.update({'figure.dpi': 130, 'font.size': 11})
sns.set_theme(style='whitegrid', palette='muted')

In [ ]:
RESULTS = {
    'Fine-tuning':   '../results/baseline/finetune_cifar100_acc_matrix.npy',
    'EWC':           '../results/baseline/ewc_cifar100_acc_matrix.npy',
    'GEM':           '../results/baseline/gem_cifar100_acc_matrix.npy',
    'PackNet':       '../results/improved/packnet_cifar100_acc_matrix.npy',
    'CausalPruning': '../results/improved/causal_cifar100_acc_matrix.npy',
}

COLORS = {
    'Fine-tuning':   '#e74c3c',
    'EWC':           '#f39c12',
    'GEM':           '#3498db',
    'PackNet':       '#2ecc71',
    'CausalPruning': '#9b59b6',
}

SPARSITY = {'Fine-tuning': 0, 'EWC': 0, 'GEM': 0, 'PackNet': 50, 'CausalPruning': 50}

matrices = {}
for name, path in RESULTS.items():
    p = Path(path)
    if p.exists():
        matrices[name] = np.load(p)
        print(f'  Loaded {name:15s}: shape={matrices[name].shape}')
    else:
        print(f'  [MISSING] {name}: {path}')

available = {k: v for k, v in matrices.items() if v is not None}

## 1. Accuracy Matrix Heatmaps

In [ ]:
n_methods = len(available)
fig, axes = plt.subplots(1, n_methods, figsize=(4 * n_methods, 5))
if n_methods == 1:
    axes = [axes]

cmaps = {'Fine-tuning': 'Reds', 'EWC': 'Oranges', 'GEM': 'Blues',
         'PackNet': 'Greens', 'CausalPruning': 'Purples'}

for ax, (name, R) in zip(axes, available.items()):
    T = R.shape[0]
    display = np.where(np.isnan(R), 0.0, R * 100)
    im = ax.imshow(display, vmin=0, vmax=100,
                   cmap=cmaps.get(name, 'YlOrRd'), aspect='auto')
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    ax.set_xticks(range(T)[::4])
    ax.set_yticks(range(T)[::4])
    ax.set_xticklabels([f'T{j}' for j in range(T)][::4], fontsize=7)
    ax.set_yticklabels([f'T{i}' for i in range(T)][::4], fontsize=7)
    ax.set_title(name, fontweight='bold', fontsize=11)
    ax.set_xlabel('Task evaluated', fontsize=9)

axes[0].set_ylabel('After training task', fontsize=9)
plt.suptitle('Accuracy Matrices — All Methods (Split-CIFAR-100, 20 tasks)',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../results/all_accuracy_matrices.png', bbox_inches='tight')
plt.show()

## 2. Forgetting Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for name, R in available.items():
    T = R.shape[0]
    # Task-0 accuracy over time
    curve = [R[i, 0] * 100 for i in range(T)]
    axes[0].plot(range(T), curve, marker='o', markersize=4,
                color=COLORS.get(name), label=name, linewidth=2)

axes[0].set_xlabel('Number of tasks trained', fontsize=12)
axes[0].set_ylabel('Accuracy on Task 0 (%)', fontsize=12)
axes[0].set_title('Catastrophic Forgetting Curves', fontsize=13, fontweight='bold')
axes[0].legend(fontsize=10); axes[0].grid(True, alpha=0.3)
axes[0].set_ylim(-5, 105)

# Final accuracy per task for all methods
x = np.arange(list(available.values())[0].shape[0])
width = 0.8 / max(len(available), 1)
for i, (name, R) in enumerate(available.items()):
    T = R.shape[0]
    final = R[T - 1, :] * 100
    axes[1].bar(x + i * width, final, width=width,
               color=COLORS.get(name), alpha=0.75, label=name)

axes[1].set_xlabel('Task index', fontsize=12)
axes[1].set_ylabel('Accuracy after all tasks (%)', fontsize=12)
axes[1].set_title('Per-task Final Accuracy', fontsize=13, fontweight='bold')
axes[1].legend(fontsize=9); axes[1].grid(True, axis='y', alpha=0.3)

plt.suptitle('Forgetting Analysis — Split-CIFAR-100', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../results/forgetting_curves.png', bbox_inches='tight')
plt.show()

## 3. BWT / FWT / Average Accuracy Comparison

In [ ]:
names = list(available.keys())
avgs  = [average_accuracy(available[n])   * 100 for n in names]
bwts  = [backward_transfer(available[n])  * 100 for n in names]
fwts  = [forward_transfer(available[n])   * 100 for n in names]
clrs  = [COLORS.get(n, 'gray')               for n in names]

x = np.arange(len(names))
w = 0.25
fig, ax = plt.subplots(figsize=(11, 5))
b1 = ax.bar(x - w,  avgs, w, label='Avg Accuracy (%)', color='#2ecc71',  alpha=0.85)
b2 = ax.bar(x,      bwts, w, label='BWT (%)',           color='#e74c3c',  alpha=0.85)
b3 = ax.bar(x + w,  fwts, w, label='FWT (%)',           color='#3498db',  alpha=0.85)

ax.set_xticks(x); ax.set_xticklabels(names, fontsize=11)
ax.set_ylabel('Score (%)', fontsize=12)
ax.set_title('Avg Accuracy / BWT / FWT — All Methods', fontsize=13, fontweight='bold')
ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
ax.legend(fontsize=10); ax.grid(True, axis='y', alpha=0.3)

for bars in [b1, b2, b3]:
    for bar in bars:
        h = bar.get_height()
        ax.annotate(f'{h:.1f}', xy=(bar.get_x() + bar.get_width()/2, h),
                    xytext=(0, 3), textcoords='offset points',
                    ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.savefig('../results/bwt_fwt_comparison.png', bbox_inches='tight')
plt.show()

# Tabulate
import pandas as pd
df = pd.DataFrame({'Method': names, 'Avg Acc (%)': avgs, 'BWT (%)': bwts, 'FWT (%)': fwts,
                   'Sparsity (%)': [SPARSITY.get(n, 0) for n in names]})
df = df.set_index('Method').round(2)
print(df.to_string())

## 4. Sparsity vs Accuracy

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
for name in names:
    sp  = SPARSITY.get(name, 0)
    acc = average_accuracy(available[name]) * 100
    ax.scatter(sp, acc, s=160, color=COLORS.get(name), zorder=5)
    ax.annotate(name, (sp, acc), textcoords='offset points',
               xytext=(7, 4), fontsize=10)

ax.set_xlabel('Network Sparsity (%)', fontsize=12)
ax.set_ylabel('Average Accuracy (%)', fontsize=12)
ax.set_title('Sparsity vs Accuracy Trade-off', fontsize=13, fontweight='bold')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../results/sparsity_vs_accuracy.png', bbox_inches='tight')
plt.show()

## 5. 4-Panel Benchmark Comparison (Hero Figure)

In [ ]:
fig = plt.figure(figsize=(14, 10))
gs  = gridspec.GridSpec(2, 2, figure=fig, hspace=0.38, wspace=0.3)

ax1 = fig.add_subplot(gs[0, 0])   # Avg accuracy bar
ax2 = fig.add_subplot(gs[0, 1])   # BWT bar
ax3 = fig.add_subplot(gs[1, 0])   # Forgetting curves
ax4 = fig.add_subplot(gs[1, 1])   # Sparsity scatter

fig.suptitle('Continual Learning Benchmark — Split-CIFAR-100 (20 tasks)',
             fontsize=15, fontweight='bold')

# Panel 1: Avg accuracy
bars = ax1.bar(x, avgs, color=clrs, alpha=0.85, width=0.55)
ax1.set_xticks(x); ax1.set_xticklabels(names, rotation=20, ha='right', fontsize=10)
ax1.set_title('Average Accuracy', fontweight='bold'); ax1.set_ylabel('Acc (%)')
ax1.grid(True, axis='y', alpha=0.3)
for bar, v in zip(bars, avgs):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
             f'{v:.1f}', ha='center', fontsize=9)

# Panel 2: BWT
bars2 = ax2.bar(x, bwts, color=clrs, alpha=0.85, width=0.55)
ax2.set_xticks(x); ax2.set_xticklabels(names, rotation=20, ha='right', fontsize=10)
ax2.set_title('Backward Transfer (BWT)', fontweight='bold'); ax2.set_ylabel('BWT (%)')
ax2.axhline(0, color='black', linewidth=0.8, linestyle='--')
ax2.grid(True, axis='y', alpha=0.3)
for bar, v in zip(bars2, bwts):
    ypos = max(v, 0) + 0.3 if v >= 0 else v - 1.5
    ax2.text(bar.get_x() + bar.get_width()/2, ypos, f'{v:.1f}', ha='center', fontsize=9)

# Panel 3: Forgetting curves
for name in names:
    R = available[name]
    T = R.shape[0]
    ax3.plot(range(T), [R[i,0]*100 for i in range(T)],
             marker='o', markersize=3, color=COLORS.get(name), label=name, linewidth=2)
ax3.set_xlabel('Tasks trained'); ax3.set_ylabel('Acc on Task 0 (%)')
ax3.set_title('Forgetting Curves (Task 0)', fontweight='bold')
ax3.legend(fontsize=9, loc='lower left'); ax3.grid(True, alpha=0.3)
ax3.set_ylim(-5, 105)

# Panel 4: Sparsity vs accuracy
for name in names:
    acc = average_accuracy(available[name]) * 100
    sp  = SPARSITY.get(name, 0)
    ax4.scatter(sp, acc, s=150, color=COLORS.get(name), zorder=5)
    ax4.annotate(name, (sp, acc), textcoords='offset points', xytext=(5, 4), fontsize=9)
ax4.set_xlabel('Sparsity (%)'); ax4.set_ylabel('Avg Accuracy (%)')
ax4.set_title('Sparsity vs Accuracy', fontweight='bold')
ax4.grid(True, alpha=0.3)

plt.savefig('../results/benchmark_comparison.png', bbox_inches='tight', dpi=150)
plt.show()
print('Hero figure saved → results/benchmark_comparison.png')

## 6. Results Summary Table

In [ ]:
import pandas as pd

rows = []
for name in names:
    R = available[name]
    rows.append({
        'Method'        : name,
        'Avg Acc (%)'   : round(average_accuracy(R)  * 100, 2),
        'BWT (%)'       : round(backward_transfer(R) * 100, 2),
        'FWT (%)'       : round(forward_transfer(R)  * 100, 2),
        'Sparsity (%)'  : SPARSITY.get(name, 0),
        'Forgetting'    : 'Yes' if backward_transfer(R) < -0.02 else 'No',
    })

df = pd.DataFrame(rows).set_index('Method')
print('='*70)
print(df.to_string())
print('='*70)
print('\nBest avg accuracy:', df["Avg Acc (%)"].idxmax())
print('Best BWT:',          df["BWT (%)"].idxmax())
print('Most efficient (acc at 50% sparsity):', df[df['Sparsity (%)'] >= 50]['Avg Acc (%)'].idxmax()
      if any(df['Sparsity (%)'] >= 50) else 'N/A')